# Exploração da camada Gold (`cinerocket.db`)

Antes de escrever o agente, conferi o banco para descobrir as armadilhas que um modelo de
linguagem cometeria ao gerar SQL. Cada achado aqui virou uma regra em
`cinedata_agent/perguntar/semantic.yaml`, um exemplo em `examples.yaml` ou um teste.

Nenhuma célula chama o LLM.

In [1]:
import sqlite3

import pandas as pd

pd.set_option("display.max_colwidth", 60)
conn = sqlite3.connect("file:../data/cinerocket.db?mode=ro", uri=True)

def q(sql):
    return pd.read_sql(sql, conn)

## 1. Tabelas e volumes

In [2]:
tabelas = q(
    "SELECT name FROM sqlite_master "
    "WHERE type = 'table' AND name != 'alembic_version' ORDER BY name"
)["name"]
contagens = [q(f'SELECT COUNT(*) AS n FROM "{t}"')["n"][0] for t in tabelas]
pd.DataFrame({"tabela": tabelas, "linhas": contagens})

,tabela,linhas
0,bridge_movie_company,116326
1,bridge_movie_genre,121521
2,bridge_movie_person,745450
3,dim_companies,45941
4,dim_genres,19
5,dim_movies,95645
6,dim_people,424656
7,dim_reviews,40267
8,fact_movies_performance,95645
9,movie_reviews,43666


O esquema difere dos CSVs da atividade anterior: `movie_reviews` usa colunas em inglês
(`name`, `rating`, `text`) e `dim_movies.idioma_original` está sempre vazia.

In [3]:
q("PRAGMA table_info('movie_reviews')")[["name", "type"]].T

,0,1,2,3,4,5,6
name,id,sk_movie_review_id,sk_movie_id,name,rating,text,created_at
type,INTEGER,VARCHAR(64),VARCHAR(64),VARCHAR(120),DOUBLE,VARCHAR(4000),DATETIME


In [4]:
q("SELECT COUNT(*) AS filmes, COUNT(idioma_original) AS com_idioma FROM dim_movies")

,filmes,com_idioma
0,95645,0


## 2. Finanças: o lucro não é zero quando falta receita

`lucro_brl = receita_brl − orcamento_brl`, com valores ausentes tratados como 0. Um filme sem
receita, mas com orçamento, aparece com **prejuízo igual ao orçamento**. Por isso as perguntas
do enunciado pedem "apenas filmes com receita informada".

In [5]:
q('''
SELECT
  SUM(receita_brl IS NOT NULL) AS com_receita,
  SUM(orcamento_brl IS NOT NULL) AS com_orcamento,
  SUM(receita_brl IS NOT NULL AND orcamento_brl IS NOT NULL) AS com_ambos,
  COUNT(*) AS total
FROM fact_movies_performance''')

,com_receita,com_orcamento,com_ambos,total
0,3373,7926,1630,95645


In [6]:
q('''
SELECT d.titulo, f.receita_brl, f.orcamento_brl, f.lucro_brl
FROM fact_movies_performance f JOIN dim_movies d USING (sk_movie_id)
WHERE f.receita_brl IS NULL ORDER BY f.lucro_brl LIMIT 5''')

,titulo,receita_brl,orcamento_brl,lucro_brl
0,Enea,None,4371002400,-4371002400
1,Red Dead Redemption 2,None,1837300000,-1837300000
2,The Marvels,None,1342370520,-1342370520
3,Guardians Of The Galaxy Vol. 3,None,1255525000,-1255525000
4,Wish,None,979240000,-979240000


### Valores lixo distorcem a margem

Há receitas de R$ 3 e orçamentos de R$ 5. Sem um piso, o ranking de margem só mostra esses
casos, e a margem média por gênero fica absurda (−534% a −152.002%).

In [7]:
q('''
SELECT d.titulo, f.receita_brl, f.orcamento_brl,
       ROUND(100.0 * f.lucro_brl / f.receita_brl, 2) AS margem_pct
FROM fact_movies_performance f JOIN dim_movies d USING (sk_movie_id)
WHERE f.receita_brl IS NOT NULL AND f.orcamento_brl IS NOT NULL
ORDER BY margem_pct DESC LIMIT 5''')

,titulo,receita_brl,orcamento_brl,margem_pct
0,"Dad, I'm Sorry",95303762.50,712.12,100.00
1,Etlb,3225800.00,161.29,100.00
2,Jailbait,23778840.80,1688.44,99.99
3,Trivikrama,52328.00,20.93,99.96
4,The Good Neighbor,486949.61,538.72,99.89


In [8]:
q('''
SELECT g.nome_genero,
       ROUND(AVG(100.0 * f.lucro_brl / f.receita_brl), 2) AS media_simples_sem_piso,
       ROUND(100.0 * SUM(CASE WHEN f.receita_brl >= 10000 AND f.orcamento_brl >= 10000
                              THEN f.lucro_brl END)
             / SUM(CASE WHEN f.receita_brl >= 10000 AND f.orcamento_brl >= 10000
                        THEN f.receita_brl END), 2)
         AS margem_agregada_com_piso
FROM fact_movies_performance f
JOIN bridge_movie_genre bg USING (sk_movie_id) JOIN dim_genres g USING (sk_genre_id)
WHERE f.receita_brl IS NOT NULL AND f.orcamento_brl IS NOT NULL
GROUP BY g.nome_genero ORDER BY margem_agregada_com_piso DESC LIMIT 6''')

,nome_genero,media_simples_sem_piso,margem_agregada_com_piso
0,Horror,-1426.08,74.67
1,Adventure,-1313.41,68.71
2,Animation,-845.31,68.29
3,Science Fiction,-3380.72,68.18
4,Family,-1335.65,67.63
5,War,-534.88,66.89


**Decisão:** margem por filme só com receita e orçamento ≥ R$ 10 mil (restam 1.521 de 1.630
filmes); margem média de um grupo = lucro total ÷ receita total (média ponderada pela
receita). Com essa definição, Terror lidera, o que bate com o que se sabe do mercado.

## 3. Popularidade corrompida em filmes sem votos

Quatro filmes quase sem votos têm popularidade igual a um ano (o número veio do título).
**Decisão:** ignorar filmes com menos de 50 votos somando TMDB e IMDb.

In [9]:
q('''
SELECT d.titulo, f.popularidade, f.qtd_tmdb, f.qtd_imdb
FROM fact_movies_performance f JOIN dim_movies d USING (sk_movie_id)
ORDER BY f.popularidade DESC LIMIT 7''')

,titulo,popularidade,qtd_tmdb,qtd_imdb
0,Blue Beetle,2994.357,1023.0,119501.0
1,Gran Turismo,2680.593,702.0,138696.0
2,La Fellinette,2020.000,2.0,NaN
3,The Fear Footage 2: Curse Of The Tape,2019.000,5.0,NaN
4,Wwe Survivor Series 2018,2018.000,17.0,NaN
5,Battipaglia 1969,1969.000,1.0,NaN
6,The Nun Ii,1692.778,NaN,84018.0


## 4. Notas: zero no TMDB significa "sem nota"

In [10]:
q('''
SELECT SUM(nota_tmdb = 0) AS tmdb_zero, SUM(nota_imdb IS NULL) AS imdb_nula,
       MIN(nota_imdb) AS imdb_min, MAX(nota_imdb) AS imdb_max
FROM fact_movies_performance''')

,tmdb_zero,imdb_nula,imdb_min,imdb_max
0,36185,12674,0.0,10.0


## 5. Datas e status

In [11]:
q("SELECT ano_lancamento, COUNT(*) AS filmes FROM dim_movies GROUP BY 1 ORDER BY 1").T

,0,1,2,3,4,5,6,7,8,9,10,11,12
ano_lancamento,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2029
filmes,12051,12987,13013,13349,10953,11052,11363,8993,1875,5,1,2,1


In [12]:
q("SELECT status_filme, COUNT(*) AS filmes FROM dim_movies GROUP BY 1 ORDER BY 2 DESC")

,status_filme,filmes
0,Lançado,94304
1,Pós-Produção,693
2,Em Produção,598
3,Planejado,50


A base vai até 2029, mas quase não há filmes depois de 2024. "Últimos 5 anos" usa o ano
atual do SQLite (`strftime('%Y', 'now')`) e só filmes com status `Lançado`.

## 6. Pessoas: o papel fica em `dim_people`

`bridge_movie_person` não tem papel; ele está em `dim_people.tipo_pessoa`. A mesma pessoa real
tem uma linha por papel (6.738 nomes aparecem como ator e como diretor).

In [13]:
q("SELECT tipo_pessoa, COUNT(*) AS pessoas FROM dim_people GROUP BY 1")

,tipo_pessoa,pessoas
0,Ator,273400
1,Diretor,65200
2,Roteirista,86056


In [14]:
q('''
SELECT COUNT(*) AS nomes_ator_e_diretor FROM (
  SELECT nome_pessoa FROM dim_people WHERE tipo_pessoa IN ('Ator', 'Diretor')
  GROUP BY nome_pessoa HAVING COUNT(DISTINCT tipo_pessoa) = 2)''')

,nomes_ator_e_diretor
0,6738


## 7. Avaliações dos usuários

`dim_reviews` agora bate 100% com `movie_reviews`. Quase todos os filmes têm uma única
avaliação, então comparações com a média dos usuários exigem pelo menos 3 avaliações.

In [15]:
q('''
WITH mr AS (SELECT sk_movie_id, COUNT(*) AS n, AVG(rating) AS m FROM movie_reviews GROUP BY 1)
SELECT COUNT(*) AS filmes,
       SUM(d.qtd_avaliacoes_usuarios = mr.n) AS quantidade_bate,
       SUM(ABS(d.nota_media_usuarios - mr.m) < 0.01) AS media_bate
FROM dim_reviews d JOIN mr USING (sk_movie_id)''')

,filmes,quantidade_bate,media_bate
0,40267,40267,40267


In [16]:
q('''
SELECT n AS avaliacoes_por_filme, COUNT(*) AS filmes
FROM (SELECT COUNT(*) AS n FROM movie_reviews GROUP BY sk_movie_id)
GROUP BY n ORDER BY n''').T

,0,1,2,3,4,5,6,7,8,9,10,11,12
avaliacoes_por_filme,1,2,3,4,5,6,7,8,9,10,11,12,13
filmes,37575,2328,243,42,21,22,14,8,7,4,1,1,1


## 8. Gêneros em inglês e títulos repetidos

In [17]:
q("SELECT GROUP_CONCAT(nome_genero, ', ') AS generos FROM dim_genres")

,generos
0,"Action, Adventure, Animation, Comedy, Crime, Documentary..."


In [18]:
q('''
SELECT
  (SELECT COUNT(*) FROM dim_movies
   WHERE sk_movie_id NOT IN (SELECT sk_movie_id FROM bridge_movie_genre)) AS filmes_sem_genero,
  (SELECT COUNT(*) FROM (SELECT 1 FROM dim_movies GROUP BY titulo, ano_lancamento
                         HAVING COUNT(*) > 1)) AS titulos_repetidos_no_mesmo_ano''')

,filmes_sem_genero,titulos_repetidos_no_mesmo_ano
0,20037,1518


## 9. Outras sujeiras da fonte (limitações conhecidas)

Vêm do próprio TMDB e não foram corrigidas, porque a camada Gold é somente leitura:

In [19]:
q('''
SELECT p.nome_pessoa AS diretor, COUNT(*) AS filmes
FROM dim_people p JOIN bridge_movie_person bp USING (sk_person_id)
WHERE p.tipo_pessoa = 'Diretor' GROUP BY p.sk_person_id ORDER BY filmes DESC LIMIT 3''')

,diretor,filmes
0,English,248
1,United States Of America,94
2,Kevin Dunn,78


Um "diretor" chamado *English* aparece em 248 filmes, e há títulos que não são filmes
(ex.: jogos e transmissões de luta livre). O agente responde com o que está na base e
cita essas limitações quando aparecem.